# **2.** Comparación estadística de Energía de señales EEG

In [1]:
import numpy as np
import pandas as pd
import scipy.io as sio
import glob
import os

#### **1.** Implemente una función que reciba una señal de múltiples canales y épocas y calcule la Energía de promedio de cada canal

In [2]:
def calcular_energia_promedio(senal):
    """
    Recibe una matriz de EEG con forma (canales, muestras, épocas).
    Retorna un arreglo 1D con la energía promedio de cada canal.
    """
    # 1. Energía por época: Sumamos los valores al cuadrado a lo largo de las muestras (axis=1)
    # Si senal es (8, 2000, 40), energia_epocas será (8, 40)
    energia_epocas = np.sum(senal**2, axis=1)
    
    # 2. Energía promedio: Promediamos esas energías a lo largo de las épocas (axis=1 en la nueva matriz)
    # El resultado será un arreglo de 8 valores (uno por canal)
    energia_promedio = np.mean(energia_epocas, axis=1)
    
    return energia_promedio

#### **2.** Calcule la energía de cada canal promediada por épocas para cada sujeto, esto para ambos grupos poblacionales. Guarde esta información en un DataFrame de columnas ‘canal’ y filas ‘#sujeto’ con los valores de energía calculados, un DataFrame para cada grupo poblacional.

In [5]:
def procesar_carpeta_eeg(ruta_carpeta, prefijo_archivo):
    """
    Lee archivos .mat de una carpeta, calcula la energía promedio de cada canal
    y retorna un DataFrame.
    """
    # Busca todos los archivos en la carpeta que coincidan con el prefijo (ej. 'p*.mat')
    patron_busqueda = os.path.join(ruta_carpeta, f'{prefijo_archivo}*.mat')
    archivos = glob.glob(patron_busqueda)
    
    datos_sujetos = []
    nombres_sujetos = []
    
    for archivo in archivos:
        # Cargar el archivo .mat
        mat_data = sio.loadmat(archivo)
        
        # Extraer la variable con los datos (ignorando las llaves internas de MATLAB que empiezan con '__')
        llaves_validas = [k for k in mat_data.keys() if not k.startswith('__')]
        llave_datos = llaves_validas[0] 
        senal_eeg = mat_data[llave_datos]
        
        # Calcular la energía pasando los datos a nuestra función
        energia_canales = calcular_energia_promedio(senal_eeg)
        datos_sujetos.append(energia_canales)
        
        # Guardar el nombre del archivo para usarlo como nombre de sujeto
        nombre_sujeto = os.path.basename(archivo).replace('.mat', '')
        nombres_sujetos.append(nombre_sujeto)
        
    # Crear el DataFrame
    num_canales = len(datos_sujetos[0]) if datos_sujetos else 0
    nombres_canales = [f'Canal_{i+1}' for i in range(num_canales)]
    
    df = pd.DataFrame(datos_sujetos, index=nombres_sujetos, columns=nombres_canales)
    df.index.name = '#sujeto'
    
    return df

# === EJECUCIÓN ===
# Importante: Asegúrate de poner la ruta correcta hacia tus carpetas
ruta_parkinson = 'C:\\Users\\HP\\Desktop\\UDEA\\2026_1\\LAB_BIOSEÑALES\\parkinson' # Reemplaza con tu ruta real si es distinta
ruta_control = 'C:\\Users\\HP\\Desktop\\UDEA\\2026_1\\LAB_BIOSEÑALES\\control'     # Reemplaza con tu ruta real si es distinta

df_parkinson = procesar_carpeta_eeg(ruta_parkinson, 'P')
df_control = procesar_carpeta_eeg(ruta_control, 'C')

# Para visualizar cómo quedó la tabla de Parkinson:
print("--- DataFrame Parkinson ---")
display(df_parkinson.head())

print("\n--- DataFrame Control ---")
display(df_control.head())

--- DataFrame Parkinson ---


,Canal_1,Canal_2,Canal_3,Canal_4,Canal_5,Canal_6,Canal_7,Canal_8
#sujeto,,,,,,,,
P001_EP_reposo,12438.243570,11261.175800,10819.634775,9489.784462,12091.060945,22798.213463,23700.620349,25606.065340
P004_EP_reposo,17995.660058,12001.601821,12286.344400,14785.908284,17058.433161,63983.449318,53715.460772,66403.639479
P005_EP_reposo,38092.102574,43575.379457,41979.994799,41715.287990,46513.737045,251649.394709,179345.438488,262361.180410
P007_EP_reposo,23742.325612,22070.007569,24540.315612,21803.936448,22594.339745,128314.264805,128888.485633,152799.284248
P012_EP_reposo,48574.518921,51806.529769,73171.952374,59707.699631,56552.175747,287105.761622,222745.793414,353312.298104



--- DataFrame Control ---


,Canal_1,Canal_2,Canal_3,Canal_4,Canal_5,Canal_6,Canal_7,Canal_8
#sujeto,,,,,,,,
C001R_EP_reposo,21465.650358,20985.907912,22760.149588,18505.640284,29730.163026,25244.158073,22781.327587,24658.599512
C002_EP_reposo,15966.402868,17617.810248,20804.937129,19654.400017,16678.982063,93894.049009,66862.496275,75685.125872
C004_EP_reposo,14148.673322,18283.999666,28749.932148,14270.726911,28787.445978,14661.417740,15940.154095,19499.898656
C005_EP_reposo_Repetido,35311.301696,34916.686010,38800.429029,35427.031127,35905.472869,106598.128152,106885.575966,112520.750636
C006_EP_reposo,18510.829979,19738.489375,20911.792748,21828.254399,23351.992649,53086.059766,37495.972475,43067.095504


#### **3.** Evaluación de diferencias estadísticas entre canales de grupos mediante una prueba t, verificando normalidad, independencia y homocedasticidad (Levene).